In [17]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.svm import SVC
from itertools import product
from sklearn.base import clone
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.decomposition import PCA
from joblib import Parallel, delayed
from scipy.stats import randint, uniform
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import KFold, cross_val_score
from sklearn.base import clone
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

In [2]:
df = pd.read_csv("../raw_data/protein_sequences/clustered_protein_sequences.csv")
df.head()

,uniprot_id,protein_name,protein_sequence
0,VM2_BOTAS,Clustered_Disintegrins,MIEVLLVTICLAVSPYQGSSIILESGNVNDYEVVYPRKVTELPKGA...
1,VM2CO_AGKCO,Clustered_Disintegrins,MIQVLLVTLCLAAFPYQGSSIILESGNVNDYEVLYPQKVTALPKGA...
2,VM3_OPHHA,Clustered_Disintegrins,MIQVLLVTICLVVFPYQGSSIILESGKVNDYEVVYPQKIPVLPKSK...
3,VM3_VIPAA,Clustered_Disintegrins,MVTKYSSIFMSPILSNPPILYFSDCSREXYQKXLTN
4,VM3_MICIK,Clustered_Disintegrins,TNTPEQDRYLQVKKYLEYVVVDNNMYRNYGNAGPCVMSAEISFEPL...


In [3]:
#class to generate k-mer based features from protein sequences

class ProteinFeatureGenerator:
    def __init__(self,
                 raw_path: str,
                 processed_path: str,
                 approaches: list = ["single", "di", "tri"]):
        self.raw_path = raw_path
        self.processed_path = processed_path
        self.approaches = approaches

        # Standard 20 amino acids
        self.amino_acids = list("ACDEFGHIKLMNPQRSTVWY")

        # Predefined k-mer combinations for 1-mer, 2-mer, and 3-mer
        self.kmer_sets = {
            "single": self.amino_acids,
            "di": [''.join(p) for p in product(self.amino_acids, repeat=2)],
            "tri": [''.join(p) for p in product(self.amino_acids, repeat=3)],
        }

        self.dataset = pd.read_csv(self.raw_path)

    # Function to remove any unknown or non-standard amino acids from a sequence
    def clean_sequence(self, seq):
        return ''.join([aa for aa in seq if aa in self.amino_acids])

    # Function to compute k-mer frequencies for a given sequence
    def compute_kmer_freq(self, seq, kmers):
        k = len(kmers[0])
        seq = self.clean_sequence(seq)
        counter = Counter([seq[i:i+k] for i in range(len(seq)-k+1)])
        total = sum(counter[k] for k in kmers)
        return [counter.get(k, 0)/total if total > 0 else 0 for k in kmers]

    # Apply k-mer frequency computation in parallel for all sequences
    def compute_features(self, sequences, kmers, desc):
        return Parallel(n_jobs=-1)(
            delayed(self.compute_kmer_freq)(seq, kmers) for seq in tqdm(sequences, desc=desc)
        )

    # Main function to generate and save feature vectors
    def generate_features(self):
        feature_frames = [self.dataset[["uniprot_id", "protein_name"]]]

        for kmer_type in self.approaches:
            if kmer_type not in self.kmer_sets:
                print(f"Warning: '{kmer_type}' is not a supported k-mer type. Skipping.")
                continue

            print(f"Generating {kmer_type}-mer frequency features...")
            kmers = self.kmer_sets[kmer_type]
            freq_matrix = self.compute_features(self.dataset["protein_sequence"], kmers, desc=f"{kmer_type.capitalize()}")
            freq_df = pd.DataFrame(freq_matrix, columns=[f"freq_{k}" for k in kmers])
            feature_frames.append(freq_df)

        full_df = pd.concat(feature_frames, axis=1)

        os.makedirs(os.path.dirname(self.processed_path), exist_ok=True)
        full_df.to_csv(self.processed_path, index=False)

        print(f"Features saved to: {self.processed_path}")


if __name__ == "__main__":
    feature_gen = ProteinFeatureGenerator(
        raw_path="../raw_data/protein_sequences/clustered_protein_sequences.csv",
        processed_path="../processed_data/protein_features/processed_protein_features.csv",
        approaches=["single", "di", "tri"]
    )

    feature_gen.generate_features()

Generating single-mer frequency features...


Single: 100%|██████████████████████████████████████████████████████████████████████| 1934/1934 [00:21<00:00, 91.73it/s]


Generating di-mer frequency features...


Di: 100%|█████████████████████████████████████████████████████████████████████████| 1934/1934 [00:09<00:00, 204.20it/s]


Generating tri-mer frequency features...


Tri: 100%|████████████████████████████████████████████████████████████████████████| 1934/1934 [00:15<00:00, 128.25it/s]


Features saved to: ../processed_data/protein_features/processed_protein_features.csv


In [4]:
    #  Split based on fold column in clustered_metadata.csv 
    print("Splitting data based on folds...")

    # Load metadata and processed features
    metadata_df = pd.read_csv("../raw_data/metadata/clustered_metadata.csv")
    features_df = pd.read_csv("../processed_data/protein_features/processed_protein_features.csv")

    # Merge on 'uniprot_id'
    merged_df = metadata_df[["uniprot_id", "fold"]].merge(features_df, on="uniprot_id", how="inner")

    # Split into train, test, and val sets
    for split in ["train", "test", "val"]:
        split_df = merged_df[merged_df["fold"] == split]
        output_path = f"../processed_data/Split data/{split}_dd.csv"
        split_df.to_csv(output_path, index=False)
        print(f"Saved {split} set to: {output_path}")

Splitting data based on folds...
Saved train set to: ../processed_data/Split data/train_dd.csv
Saved test set to: ../processed_data/Split data/test_dd.csv
Saved val set to: ../processed_data/Split data/val_dd.csv
